# 07 — LLM Prompt Engineering & Hallucination Reduction

Experiments with prompt design for the credibility report generator.

**Key techniques demonstrated:**
1. System prompt constraints (no fabrication rules)
2. Structured output enforcement (JSON schema)
3. Chain-of-thought internally, structured output externally
4. Uncertainty handling (`"Insufficient evidence"` fallback)
5. Temperature ablation (high temp → hallucination, low temp → grounded)

## 1. Setup

In [ ]:
import sys, json, os
sys.path.insert(0, '../backend')

from agent.llm_agent import _call_hf_api, _extract_json, _rule_based_report, SYSTEM_PROMPT, REPORT_PROMPT_TEMPLATE
from agent.predictor import predict
from agent.risk_analyzer import analyze_risk
from agent.retriever import retrieve

HF_TOKEN_SET = bool(os.getenv('HF_TOKEN', ''))
print(f'HF_TOKEN set: {HF_TOKEN_SET}')
print('If False, all LLM cells will show rule-based fallback output.')

## 2. The System Prompt — Anti-Hallucination Rules

In [ ]:
print(SYSTEM_PROMPT)

### Why these rules work:

| Rule | Hallucination it prevents |
|---|---|
| `NEVER fabricate facts` | Invented sources, fake quotes |
| `"Insufficient evidence"` fallback | Confident-sounding wrong answers |
| Separate facts vs assessments | Mixing model output with LLM invention |
| Structured JSON output | Free-form text that invents narrative |
| Low temperature (0.1) | Random/creative outputs |

## 3. Prompt Experiment A — Baseline (no constraints)

In [ ]:
ARTICLE = """
BREAKING: Scientists discover miracle cure for all diseases hidden by Big Pharma for decades.
Anonymous sources confirm the government has been suppressing this information. Share before deleted!
""".strip()

# Baseline prompt — no constraints
baseline_prompt = f"""Analyze this news article and tell me if it's real or fake:

{ARTICLE}

Give me a detailed analysis."""

print('Baseline prompt (no constraints):')
print(baseline_prompt)
print()
print('Problem: LLM may fabricate sources, invent statistics, or give inconsistent verdicts.')

## 4. Prompt Experiment B — With Constraints + Structured Output

In [ ]:
pred = predict(ARTICLE)
risk = analyze_risk(ARTICLE)
docs = retrieve(f"{pred['label']} {' '.join(pred['top_features'][:5])}", top_k=3)

retrieved_context = '\n'.join(
    f"[{d['source']}] {d['title']}: {d['content'][:200]}"
    for d in docs
)

constrained_prompt = REPORT_PROMPT_TEMPLATE.format(
    system_prompt=SYSTEM_PROMPT,
    article_excerpt=ARTICLE[:500],
    label=pred['label'],
    confidence=pred['confidence'],
    confidence_tier=pred['confidence_tier'],
    real_prob=pred['real_probability'],
    fake_prob=pred['fake_probability'],
    top_features=', '.join(pred['top_features'][:8]),
    risk_score=risk['risk_score'],
    risk_factors='; '.join(risk['risk_factors']) or 'None',
    credibility_indicators='; '.join(risk['credibility_indicators']) or 'None',
    retrieved_context=retrieved_context,
)

print('Constrained prompt (first 800 chars):')
print(constrained_prompt[:800])
print('...')

## 5. Call LLM (requires HF_TOKEN)

In [ ]:
if HF_TOKEN_SET:
    print('Calling HuggingFace API (Mistral-7B-Instruct)...')
    raw_output = _call_hf_api(constrained_prompt)
    print('Raw LLM output:')
    print(raw_output)
else:
    print('HF_TOKEN not set — showing rule-based fallback output instead.')
    print('To use LLM: export HF_TOKEN=hf_your_token_here')
    print('Get free token at: https://huggingface.co/settings/tokens')

## 6. Parse & Validate JSON Output

In [ ]:
# Test JSON extraction on a sample LLM-style output
sample_llm_output = '''Sure! Here is the analysis:
```json
{
  "summary": "The ML model classified this as Fake News with 91.2% confidence. Multiple clickbait patterns and conspiracy language were detected.",
  "credibility_indicators": ["No credibility indicators detected"],
  "risk_factors": ["Clickbait language detected", "Anonymous sources only", "Conspiracy framing"],
  "cross_source_verification": "Retrieved context confirms conspiracy theory language patterns match known misinformation templates.",
  "confidence_assessment": "91.2% confidence is high-tier. The model is reliable at this confidence level.",
  "sources": ["First Draft News", "Snopes Methodology"],
  "disclaimer": "This analysis is AI-generated and should not be the sole basis for determining truth."
}
```'''

parsed = _extract_json(sample_llm_output)
if parsed:
    print('JSON extracted successfully:')
    print(json.dumps(parsed, indent=2))
else:
    print('JSON extraction failed')

## 7. Rule-Based Fallback Demo

In [ ]:
# Demonstrate the deterministic fallback — zero hallucination guaranteed
state = {
    'raw_text': ARTICLE,
    'prediction': pred,
    'risk_analysis': risk,
    'retrieved_docs': docs,
}

fallback_report = _rule_based_report(state)
print('Rule-based fallback report (no LLM, no hallucination):')
print(json.dumps(fallback_report, indent=2))

## 8. Temperature Ablation — Hallucination Risk

In [ ]:
import pandas as pd

temperature_analysis = pd.DataFrame([
    {'temperature': 0.0, 'behavior': 'Deterministic, most grounded', 'hallucination_risk': 'Very Low', 'use_case': 'Fact-checking'},
    {'temperature': 0.1, 'behavior': 'Near-deterministic, slight variation', 'hallucination_risk': 'Low', 'use_case': 'Our system (default)'},
    {'temperature': 0.5, 'behavior': 'Balanced creativity/accuracy', 'hallucination_risk': 'Medium', 'use_case': 'General Q&A'},
    {'temperature': 0.9, 'behavior': 'Creative, unpredictable', 'hallucination_risk': 'High', 'use_case': 'Creative writing'},
    {'temperature': 1.2, 'behavior': 'Chaotic, often incoherent', 'hallucination_risk': 'Very High', 'use_case': 'Not recommended'},
])

print(temperature_analysis.to_string(index=False))
print('\nOur system uses temperature=0.1 to minimize hallucination.')

## 9. Uncertainty Handling — "Insufficient Evidence" Test

In [ ]:
# Short, ambiguous article — should trigger uncertainty
AMBIGUOUS = 'Something happened today. People were there. It was notable.'

pred_amb = predict(AMBIGUOUS)
risk_amb = analyze_risk(AMBIGUOUS)

print(f'Confidence: {pred_amb["confidence"]}% ({pred_amb["confidence_tier"]})')
print(f'Word count: {pred_amb["word_count"]}')
print(f'Risk score: {risk_amb["risk_score"]}')
print()

if pred_amb['confidence_tier'] == 'low' or pred_amb['word_count'] < 20:
    print('UNCERTAINTY FLAG: Insufficient evidence — result unreliable.')
    print('The system will include this warning in the report.')
else:
    print('Confidence sufficient for analysis.')

## Summary — Hallucination Reduction Techniques

| Technique | Implementation | Effect |
|---|---|---|
| System prompt rules | `SYSTEM_PROMPT` in `llm_agent.py` | Prevents fabricated facts |
| Structured JSON output | Schema in prompt | Prevents free-form invention |
| Low temperature (0.1) | API parameter | Reduces randomness |
| Data-grounded prompt | All facts injected | LLM can only use provided data |
| Rule-based fallback | `_rule_based_report()` | Zero hallucination guarantee |
| Uncertainty flags | Confidence tier check | Explicit "insufficient evidence" |
| JSON validation | `_extract_json()` | Rejects malformed outputs |